### Purpose

- This script adds **Stadtteil** information as an additional column to the original dataset.
- **Goal**: Enrich Karlsruhe Baustellen‑Vorschau data with district context for analysis.
- **Inputs**: Five CSVs (Point, LineString, MultiLineString, Polygon, MultiPolygon) plus the Stadtteile shapefile.
- **Geometry conversion**: Parse stored coordinates into Shapely geometries and create GeoDataFrames in `EPSG:25832`.
- **Spatial join**: Use `gpd.sjoin(..., predicate="intersects")` to attach `Stadtteil` and `Stadtteil_Nummer`.
- **Outputs**: Five `*_stadtteil` GeoDataFrames, one per geometry type, ready for analysis or export.


In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
import os
import ast
import json
from shapely.geometry import Point, shape

In [2]:
# Define input, output paths
current_dir = globals().get('current_dir', os.getcwd())
input_path = os.path.join(current_dir, '../data/mobidata/raw/csv_files/')
map_path = os.path.join(current_dir, '../data/map/Stadtteile_SHP/Stadtviertel_Karlsruhe.shp')
output_path = os.path.join(current_dir, '../data/mobidata/processed/')

# Define file path
input_file_point = 'baustellen_vorschau_Point.csv'
input_file_path_point = os.path.join(input_path, input_file_point)

input_file_polygon = 'baustellen_vorschau_Polygon.csv'
input_file_path_polygon = os.path.join(input_path, input_file_polygon)

input_file_LineString = 'baustellen_vorschau_LineString.csv'
input_file_path_LineString = os.path.join(input_path, input_file_LineString)

input_file_MultiLineString = 'baustellen_vorschau_MultiLineString.csv'
input_file_path_MultiLineString = os.path.join(input_path, input_file_MultiLineString)

input_file_MultiPolygon = 'baustellen_vorschau_MultiPolygon.csv'
input_file_path_MultiPolygon = os.path.join(input_path, input_file_MultiPolygon)

### 1. Load data

In [3]:
# Load data
df_Point = pd.read_csv(input_file_path_point)
df_Polygon = pd.read_csv(input_file_path_polygon)
df_LineString = pd.read_csv(input_file_path_LineString)
df_MultiLineString = pd.read_csv(input_file_path_MultiLineString)
df_MultiPolygon = pd.read_csv(input_file_path_MultiPolygon)

In [4]:
# Check data
df_Point.head()


,type,totalFeatures,numberMatched,numberReturned,timeStamp,crs_type,crs_properties_name,feature_type,feature_id,geometry_name,...,art,lage,tagesbaustelle,verursacher,zusatzinfo,sperrung,projektnummer,vorgangsnummer,datenquelle,stand
0,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.1112,geom,...,AK5_encours,D105 Hésingue/St-Louis,NaN,NaN,Dévoiement de la chaussée et changements des l...,NaN,6769,NaN,Collectivité européenne d’Alsace,2025-10-05T22:00:00Z
1,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.1117,geom,...,CE3a Information,"D130 - Bœrsch, Grendelbruch, Le Hohwald, Natzw...",NaN,NaN,Manifestation sportive intitulée “Biathlon <br...,NaN,6795,NaN,Collectivité européenne d’Alsace,2025-10-05T22:00:00Z
2,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.1141,geom,...,KC1_route-barree,D426 - Entre Obernai et Ottrott,NaN,NaN,Route barrée avec mise en place d'une déviatio...,NaN,6810,NaN,Collectivité européenne d’Alsace,2025-10-05T22:00:00Z
3,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.1142,geom,...,Travaux,D426 - Gerstheim,NaN,NaN,Veuillez vous référer aux documents joints pou...,NaN,6805,NaN,Collectivité européenne d’Alsace,2025-10-05T22:00:00Z
4,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.1148,geom,...,KC1_route-barree,D505 Soultz-Haut-Rhin,NaN,NaN,Travaux de voirie;<br />Veuillez consulter les...,NaN,6813,NaN,Collectivité européenne d’Alsace,2025-10-05T22:00:00Z


In [5]:
print(df_Point.keys())

Index(['type', 'totalFeatures', 'numberMatched', 'numberReturned', 'timeStamp',
       'crs_type', 'crs_properties_name', 'feature_type', 'feature_id',
       'geometry_name', 'geometry_type', 'geometry_coordinates', 'geometry_x',
       'geometry_y', 'id', 'gemeinde', 'vorgangszeitraum_von',
       'vorgangszeitraum_bis', 'art', 'lage', 'tagesbaustelle', 'verursacher',
       'zusatzinfo', 'sperrung', 'projektnummer', 'vorgangsnummer',
       'datenquelle', 'stand'],
      dtype='object')


In [6]:
df_Point['geometry_coordinates'].iloc[0]

'[389426.38255268, 5270752.73090236]'

In [7]:
# Load map
gdf_stadtteile = gpd.read_file(map_path)
# Rename columns for clarity
gdf_stadtteile = gdf_stadtteile.rename(columns={
    "NAME": "Stadtteil",
    "NUMMER": "Stadtteil_Nummer"
})

In [8]:
# Check map
gdf_stadtteile

,OBJECTID,Stadtteil_Nummer,Stadtteil,BEVD,SHAPE_AREA,SHAPE_LEN,geometry
0,47,132,Bulach,0.0,2.349710e+06,8835.931194,"POLYGON ((454615.549 5426795.403, 454620.722 5..."
1,48,122,Waldlage,0.0,6.152226e+05,3868.385783,"POLYGON ((453802.311 5426791.226, 453799.052 5..."
2,49,115,Neue Heidenstückersiedlung,0.0,7.117038e+05,4693.388222,"POLYGON ((453219.232 5427203.653, 453219.566 5..."
3,50,112,Hardecksiedlung,0.0,4.739133e+05,3617.545619,"POLYGON ((453910.998 5427616.596, 453907.044 5..."
4,51,111,Alt-Grünwinkel,0.0,1.141327e+06,4960.769418,"POLYGON ((453643.475 5428138.041, 453640.67 54..."
...,...,...,...,...,...,...,...
65,66,071,Nördlicher Teil,0.0,1.377227e+06,5216.090139,"POLYGON ((458149.345 5430144.52, 458151.172 54..."
66,67,195,Aue,0.0,2.187323e+06,7553.954377,"POLYGON ((460626.689 5425626.578, 460624.636 5..."
67,68,191,Alt-Durlach,0.0,5.583243e+06,13576.175961,"POLYGON ((462708.966 5428419.393, 462706.385 5..."
68,69,161,Waldlage,0.0,9.477397e+06,14302.383936,"POLYGON ((460151.186 5432841.279, 460150.673 5..."


### 2. Convert dataframe to geodataframe

In [9]:
# 1. Convert Point
gdf_Point = gpd.GeoDataFrame(
    df_Point,
    geometry=gpd.points_from_xy(df_Point["geometry_x"], df_Point["geometry_y"]),
    crs="EPSG:25832"
)

In [10]:
gdf_Point.head()

,type,totalFeatures,numberMatched,numberReturned,timeStamp,crs_type,crs_properties_name,feature_type,feature_id,geometry_name,...,lage,tagesbaustelle,verursacher,zusatzinfo,sperrung,projektnummer,vorgangsnummer,datenquelle,stand,geometry
0,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.1112,geom,...,D105 Hésingue/St-Louis,NaN,NaN,Dévoiement de la chaussée et changements des l...,NaN,6769,NaN,Collectivité européenne d’Alsace,2025-10-05T22:00:00Z,POINT (389426.383 5270752.731)
1,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.1117,geom,...,"D130 - Bœrsch, Grendelbruch, Le Hohwald, Natzw...",NaN,NaN,Manifestation sportive intitulée “Biathlon <br...,NaN,6795,NaN,Collectivité européenne d’Alsace,2025-10-05T22:00:00Z,POINT (373668.451 5366749.022)
2,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.1141,geom,...,D426 - Entre Obernai et Ottrott,NaN,NaN,Route barrée avec mise en place d'une déviatio...,NaN,6810,NaN,Collectivité européenne d’Alsace,2025-10-05T22:00:00Z,POINT (385210.785 5368673.758)
3,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.1142,geom,...,D426 - Gerstheim,NaN,NaN,Veuillez vous référer aux documents joints pou...,NaN,6805,NaN,Collectivité européenne d’Alsace,2025-10-05T22:00:00Z,POINT (405143.429 5359769.028)
4,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.1148,geom,...,D505 Soultz-Haut-Rhin,NaN,NaN,Travaux de voirie;<br />Veuillez consulter les...,NaN,6813,NaN,Collectivité européenne d’Alsace,2025-10-05T22:00:00Z,POINT (366913.581 5304229.635)


In [11]:
# 2. Convert Polygon
df_Polygon['geometry'] = df_Polygon.apply(
    lambda row: shape({
        'type': row['geometry_type'],
        'coordinates': ast.literal_eval(row['geometry_coordinates'])
    }),
    axis=1
)
gdf_Polygon = gpd.GeoDataFrame(df_Polygon, geometry='geometry', crs="EPSG:25832")
gdf_Polygon.head()

,type,totalFeatures,numberMatched,numberReturned,timeStamp,crs_type,crs_properties_name,feature_type,feature_id,geometry_name,...,lage,tagesbaustelle,verursacher,zusatzinfo,sperrung,projektnummer,vorgangsnummer,datenquelle,stand,geometry
0,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.101725969,geom,...,Herrenstraße zw. Karlstor und Amalienstraße,NaN,Stadtwerke Karlsruhe,Aufteilung nicht vorgenommen,mit Verkehrsbehinderung,22525,2019V3692,Stadt Karlsruhe,2025-07-27T22:00:00Z,"POLYGON ((455848.977 5428434.554, 455750.691 5..."
1,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102244740,geom,...,Reinhold-Frank-Straße zwischen Belfortstraße u...,NaN,Stadtwerke Karlsruhe,"Abwicklung in mehreren Bauabschnitten, Aufteil...",mit Sperrung in eine Fahrtrichtung,40847,2024V5284,Stadt Karlsruhe,2025-10-05T22:00:00Z,"POLYGON ((455200.314 5428615.967, 455198.747 5..."
2,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102132432,geom,...,Johann-Strauß-Straße zw. Anton-Bruckner-Straße...,NaN,Tiefbauamt,"Abwicklung in mehreren Bauabschnitten, Aufteil...",mit Verkehrsbehinderung,12438,2022V5259,Stadt Karlsruhe,2025-05-19T22:00:00Z,"POLYGON ((460090.835 5427778.357, 460168.62 54..."
3,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102135581,geom,...,Siemensallee zwischen B36/ Neureuter Straße un...,NaN,Stadtwerke Karlsruhe,"Abwicklung in mehreren Bauabschnitten, Aufteil...",mit Sperrung in eine Fahrtrichtung,40810,2022V5679,Stadt Karlsruhe,2025-09-16T22:00:00Z,"POLYGON ((452992.451 5430304.864, 452988.969 5..."
4,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102248527,geom,...,K 9675 zw. AS3 und AS4 (Edeltrudtunnel),NaN,Tiefbauamt,Ausführung jeweils zw. 22.00 und 5.00 Uhr,mit Vollsperrung,44411,2024V5696,Stadt Karlsruhe,2025-09-24T22:00:00Z,"POLYGON ((454935.672 5426740.063, 454928.478 5..."


In [12]:
# Check the structure of Polygon data
print("Columns:", df_Polygon.columns.tolist())
print("\nFirst geometry_coordinates:")
print(df_Polygon['geometry_coordinates'].iloc[0])
print("\nType after literal_eval:")
print(type(ast.literal_eval(df_Polygon['geometry_coordinates'].iloc[0])))

Columns: ['type', 'totalFeatures', 'numberMatched', 'numberReturned', 'timeStamp', 'crs_type', 'crs_properties_name', 'feature_type', 'feature_id', 'geometry_name', 'geometry_type', 'geometry_coordinates', 'geometry_x', 'geometry_y', 'id', 'gemeinde', 'vorgangszeitraum_von', 'vorgangszeitraum_bis', 'art', 'lage', 'tagesbaustelle', 'verursacher', 'zusatzinfo', 'sperrung', 'projektnummer', 'vorgangsnummer', 'datenquelle', 'stand', 'geometry']

First geometry_coordinates:
[[[455848.9770598, 5428434.5537653], [455750.69134588, 5428304.55376538], [455758.97705987, 5428298.26805139], [455859.54848779, 5428425.69662231], [455848.9770598, 5428434.5537653]]]

Type after literal_eval:
<class 'list'>


In [13]:
# 3. Convert LineString
df_LineString['geometry'] = df_LineString.apply(
    lambda row: shape({
        'type': row['geometry_type'],
        'coordinates': ast.literal_eval(row['geometry_coordinates'])
    }),
    axis=1
)
gdf_LineString = gpd.GeoDataFrame(df_LineString, geometry='geometry', crs="EPSG:25832")
gdf_LineString.head()

,type,totalFeatures,numberMatched,numberReturned,timeStamp,crs_type,crs_properties_name,feature_type,feature_id,geometry_name,...,lage,tagesbaustelle,verursacher,zusatzinfo,sperrung,projektnummer,vorgangsnummer,datenquelle,stand,geometry
0,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102302855,geom,...,Glümerstraße zw. Hardtstraße und Geibelstraße,NaN,Stadtwerke Karlsruhe,Aufteilung nicht vorgenommen,mit Verkehrsbehinderung,52214,2025V4239,Stadt Karlsruhe,2025-09-01T22:00:00Z,"LINESTRING (453184.626 5429174.898, 453188.198..."
1,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102302989,geom,...,Im Weiherwald zw. Battstraße und Lohwiesenweg,NaN,Forstamt,NaN,mit Vollsperrung,52486,2025V4261,Stadt Karlsruhe,2025-08-21T22:00:00Z,"LINESTRING (454287.823 5425235.761, 454490.787..."
2,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102305970,geom,...,L623 zw. Steinkreuzsstraße und Hohenwettersbac...,NaN,Tiefbauamt,Nächtliche Vollsperrung vs. ab 0:00 - 5:00 Uhr,mit Vollsperrung,49247,2025V4635,Stadt Karlsruhe,2025-08-13T22:00:00Z,"LINESTRING (462945.591 5426090.561, 462971.004..."
3,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102305971,geom,...,L623 zw. Steinkreuzsstraße und Hohenwettersbac...,NaN,Tiefbauamt,Nächtliche Vollsperrung vs. ab 0:00 - 5:00 Uhr,mit Vollsperrung,49247,2025V4635,Stadt Karlsruhe,2025-08-13T22:00:00Z,"LINESTRING (460746.25 5425573.74, 460772.555 5..."
4,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102310926,geom,...,Oberfeldsraße zw. Pralistraße und Welschneureu...,NaN,Stadtwerke Karlsruhe,NaN,mit Verkehrsbehinderung,47814,2025V5276,Stadt Karlsruhe,2025-09-14T22:00:00Z,"LINESTRING (454132.223 5432567.411, 454133.47 ..."


In [14]:
# 4. Convert MultiLineString
df_MultiLineString['geometry'] = df_MultiLineString.apply(
    lambda row: shape({
        'type': row['geometry_type'],
        'coordinates': ast.literal_eval(row['geometry_coordinates'])
    }),
    axis=1
)
gdf_MultiLineString = gpd.GeoDataFrame(df_MultiLineString, geometry='geometry', crs="EPSG:25832")
gdf_MultiLineString.head()

,type,totalFeatures,numberMatched,numberReturned,timeStamp,crs_type,crs_properties_name,feature_type,feature_id,geometry_name,...,lage,tagesbaustelle,verursacher,zusatzinfo,sperrung,projektnummer,vorgangsnummer,datenquelle,stand,geometry
0,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102193847,geom,...,Kloserstraße zw. Schnetzlerstraße und Gutschst...,NaN,Stadtwerke Karlsruhe,Abwicklung in mehreren Bauabschnitten,mit Verkehrsbehinderung,10273,2023V5734,Stadt Karlsruhe,2025-09-10T22:00:00Z,"MULTILINESTRING ((455923.63 5427422.033, 45592..."
1,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102193848,geom,...,Kloserstraße zw. Schnetzlerstraße und Gutschst...,NaN,Stadtwerke Karlsruhe,Abwicklung in mehreren Bauabschnitten,mit Verkehrsbehinderung,10273,2023V5734,Stadt Karlsruhe,2025-09-10T22:00:00Z,"MULTILINESTRING ((456048.834 5427160.722, 4560..."
2,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102296302,geom,...,Wendtstraße zw. Kaiserallee und Ludwig-Marum-S...,NaN,Stadtwerke Karlsruhe,"Abwicklung in mehreren Bauabschnitten, Aufteil...",mit Verkehrsbehinderung,45182,2025V3496,Stadt Karlsruhe,2025-09-24T22:00:00Z,"MULTILINESTRING ((454148.833 5429127.463, 4541..."
3,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102311078,geom,...,"Gänsbergstr., Eichwaldstr., Wiesentalstr. u. H...",NaN,Stadtwerke Karlsruhe,Aufteilung nicht vorgenommen,mit Verkehrsbehinderung,46170,2025V5298,Stadt Karlsruhe,2025-09-15T22:00:00Z,"MULTILINESTRING ((464167.538 5422461.625, 4641..."
4,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102312472,geom,...,Johann-Strauß-Straße zw. Parkplatz Lorzingstra...,NaN,Stadtwerke Karlsruhe,NaN,mit Vollsperrung,41073,2025V5450,Stadt Karlsruhe,2025-09-21T22:00:00Z,"MULTILINESTRING ((460050.149 5427790.847, 4600..."


In [15]:
# 5. Convert MultiPolygon
df_MultiPolygon['geometry'] = df_MultiPolygon.apply(
    lambda row: shape({
        'type': row['geometry_type'],
        'coordinates': ast.literal_eval(row['geometry_coordinates'])
    }),
    axis=1
)
gdf_MultiPolygon = gpd.GeoDataFrame(df_MultiPolygon, geometry='geometry', crs="EPSG:25832")
gdf_MultiPolygon.head()

,type,totalFeatures,numberMatched,numberReturned,timeStamp,crs_type,crs_properties_name,feature_type,feature_id,geometry_name,...,lage,tagesbaustelle,verursacher,zusatzinfo,sperrung,projektnummer,vorgangsnummer,datenquelle,stand,geometry
0,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102021977,geom,...,Neckarstraße zwischen Enzstraße und Dreisamstraße,NaN,Tiefbauamt,"Abwicklung in mehreren Bauabschnitten, Aufteil...",mit Verkehrsbehinderung,31027,2020V5592,Stadt Karlsruhe,2025-08-10T22:00:00Z,"MULTIPOLYGON (((455568.34 5426039.452, 455574...."
1,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102078846,geom,...,Litzenhardtsatrße zw. St.-Florian-Straße und B...,NaN,Tiefbauamt,"Abwicklung in mehreren Bauabschnitten, Aufteil...",mit Vollsperrung,35548,2021V5742,Stadt Karlsruhe,2025-09-08T22:00:00Z,"MULTIPOLYGON (((454968.865 5426532.349, 454963..."
2,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102260990,geom,...,"Breite Straße zw, Neubruchweg und Wartburgstraße",NaN,Telekommunikation,"Abwicklung in mehreren Bauabschnitten, Arbeits...",mit Verkehrsbehinderung,43211,2024V7262,Stadt Karlsruhe,2024-12-05T23:00:00Z,"MULTIPOLYGON (((454826.021 5426886.497, 454829..."
3,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102261447,geom,...,Reinhold-Frank-Straße zwischen Belfortstraße u...,NaN,Stadtwerke Karlsruhe,"Abwicklung in mehreren Bauabschnitten, Aufteil...",mit Sperrung in eine Fahrtrichtung,40847,2024V5284,Stadt Karlsruhe,2025-10-05T22:00:00Z,"MULTIPOLYGON (((455206.909 5428856.375, 455202..."
4,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102286477,geom,...,Hübschstraße zw. Kriegsstraße u. Eisenlohrstraße,NaN,Stadtwerke Karlsruhe,"Abwicklung in mehreren Bauabschnitten, Aufteil...",mit Verkehrsbehinderung,46078,2025V2418,Stadt Karlsruhe,2025-08-31T22:00:00Z,"MULTIPOLYGON (((454101.097 5428233.81, 454115...."


### 3. Add Stadtteil info to the geo dataframes

In [16]:
gdf_stadtteile_map = gdf_stadtteile[["Stadtteil_Nummer","Stadtteil", "geometry"]]

In [17]:
# 1. Add Stadtteil info to gdf_Point
gdf_Point_stadtteil = gpd.sjoin(gdf_Point, gdf_stadtteile_map, how="left", predicate="intersects")

In [18]:
gdf_Point_stadtteil

,type,totalFeatures,numberMatched,numberReturned,timeStamp,crs_type,crs_properties_name,feature_type,feature_id,geometry_name,...,zusatzinfo,sperrung,projektnummer,vorgangsnummer,datenquelle,stand,geometry,index_right,Stadtteil_Nummer,Stadtteil
0,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.1112,geom,...,Dévoiement de la chaussée et changements des l...,NaN,6769,NaN,Collectivité européenne d’Alsace,2025-10-05T22:00:00Z,POINT (389426.383 5270752.731),NaN,NaN,NaN
1,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.1117,geom,...,Manifestation sportive intitulée “Biathlon <br...,NaN,6795,NaN,Collectivité européenne d’Alsace,2025-10-05T22:00:00Z,POINT (373668.451 5366749.022),NaN,NaN,NaN
2,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.1141,geom,...,Route barrée avec mise en place d'une déviatio...,NaN,6810,NaN,Collectivité européenne d’Alsace,2025-10-05T22:00:00Z,POINT (385210.785 5368673.758),NaN,NaN,NaN
3,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.1142,geom,...,Veuillez vous référer aux documents joints pou...,NaN,6805,NaN,Collectivité européenne d’Alsace,2025-10-05T22:00:00Z,POINT (405143.429 5359769.028),NaN,NaN,NaN
4,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.1148,geom,...,Travaux de voirie;<br />Veuillez consulter les...,NaN,6813,NaN,Collectivité européenne d’Alsace,2025-10-05T22:00:00Z,POINT (366913.581 5304229.635),NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
134,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102314047,geom,...,Entfall von Parkmöglichkeiten,keine Verkehrsbehinderung,50933,2025V5267,Stadt Karlsruhe,2025-09-28T22:00:00Z,POINT (456594.105 5427575.979),16.0,032,Südlicher Teil
135,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102314476,geom,...,"9:00 Uhr - 15:00 Uhr, beide Fahrtrichtungen be...",mit Verkehrsbehinderung,49248,2025V5721,Stadt Karlsruhe,2025-09-30T22:00:00Z,POINT (451823.121 5426902.05),13.0,094,Rheinstrandsiedlung
136,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102314525,geom,...,NaN,mit Vollsperrung,53234,2025V5726,Stadt Karlsruhe,2025-09-30T22:00:00Z,POINT (450461.305 5428702.401),49.0,083,Rheinhafen
137,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102314548,geom,...,NaN,mit Vollsperrung,45072,2025V5729,Stadt Karlsruhe,2025-09-30T22:00:00Z,POINT (456711.478 5427524.471),16.0,032,Südlicher Teil


In [19]:
# 2. Add Stadtteil info to gdf_Polygon
gdf_Polygon_stadtteil = gpd.sjoin(gdf_Polygon, gdf_stadtteile_map, how="left", predicate="intersects")

In [20]:
gdf_Polygon_stadtteil

,type,totalFeatures,numberMatched,numberReturned,timeStamp,crs_type,crs_properties_name,feature_type,feature_id,geometry_name,...,zusatzinfo,sperrung,projektnummer,vorgangsnummer,datenquelle,stand,geometry,index_right,Stadtteil_Nummer,Stadtteil
0,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.101725969,geom,...,Aufteilung nicht vorgenommen,mit Verkehrsbehinderung,22525,2019V3692,Stadt Karlsruhe,2025-07-27T22:00:00Z,"POLYGON ((455848.977 5428434.554, 455750.691 5...",14,021,Östlicher Teil
1,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102244740,geom,...,"Abwicklung in mehreren Bauabschnitten, Aufteil...",mit Sperrung in eine Fahrtrichtung,40847,2024V5284,Stadt Karlsruhe,2025-10-05T22:00:00Z,"POLYGON ((455200.314 5428615.967, 455198.747 5...",43,022,Westlicher Teil
1,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102244740,geom,...,"Abwicklung in mehreren Bauabschnitten, Aufteil...",mit Sperrung in eine Fahrtrichtung,40847,2024V5284,Stadt Karlsruhe,2025-10-05T22:00:00Z,"POLYGON ((455200.314 5428615.967, 455198.747 5...",34,052,Südlicher Teil
2,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102132432,geom,...,"Abwicklung in mehreren Bauabschnitten, Aufteil...",mit Verkehrsbehinderung,12438,2022V5259,Stadt Karlsruhe,2025-05-19T22:00:00Z,"POLYGON ((460090.835 5427778.357, 460168.62 54...",54,192,Dornwald-Untermühl
3,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102135581,geom,...,"Abwicklung in mehreren Bauabschnitten, Aufteil...",mit Sperrung in eine Fahrtrichtung,40810,2022V5679,Stadt Karlsruhe,2025-09-16T22:00:00Z,"POLYGON ((452992.451 5430304.864, 452988.969 5...",50,082,Weingärtensiedlung
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
97,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102313485,geom,...,NaN,mit Vollsperrung,46572,2025V5605,Stadt Karlsruhe,2025-09-25T22:00:00Z,"POLYGON ((450732.873 5428526.651, 450728.586 5...",42,091,Alt-Daxlanden
98,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102314524,geom,...,NaN,mit Vollsperrung,53234,2025V5726,Stadt Karlsruhe,2025-09-30T22:00:00Z,"POLYGON ((450449.38 5428716.489, 450444.192 54...",42,091,Alt-Daxlanden
98,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102314524,geom,...,NaN,mit Vollsperrung,53234,2025V5726,Stadt Karlsruhe,2025-09-30T22:00:00Z,"POLYGON ((450449.38 5428716.489, 450444.192 54...",49,083,Rheinhafen
99,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102314547,geom,...,NaN,mit Vollsperrung,45072,2025V5729,Stadt Karlsruhe,2025-09-30T22:00:00Z,"POLYGON ((456710.761 5427527.977, 456705.761 5...",16,032,Südlicher Teil


In [21]:
# 3. Add Stadtteil info to gdf_LineString
gdf_LineString_stadtteil = gpd.sjoin(gdf_LineString, gdf_stadtteile_map, how="left", predicate="intersects")

In [22]:
gdf_LineString_stadtteil

,type,totalFeatures,numberMatched,numberReturned,timeStamp,crs_type,crs_properties_name,feature_type,feature_id,geometry_name,...,zusatzinfo,sperrung,projektnummer,vorgangsnummer,datenquelle,stand,geometry,index_right,Stadtteil_Nummer,Stadtteil
0,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102302855,geom,...,Aufteilung nicht vorgenommen,mit Verkehrsbehinderung,52214,2025V4239,Stadt Karlsruhe,2025-09-01T22:00:00Z,"LINESTRING (453184.626 5429174.898, 453188.198...",64,081,Alt-Mühlburg
1,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102302989,geom,...,NaN,mit Vollsperrung,52486,2025V4261,Stadt Karlsruhe,2025-08-21T22:00:00Z,"LINESTRING (454287.823 5425235.761, 454490.787...",29,141,Weiherfeld
1,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102302989,geom,...,NaN,mit Vollsperrung,52486,2025V4261,Stadt Karlsruhe,2025-08-21T22:00:00Z,"LINESTRING (454287.823 5425235.761, 454490.787...",0,132,Bulach
2,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102305970,geom,...,Nächtliche Vollsperrung vs. ab 0:00 - 5:00 Uhr,mit Vollsperrung,49247,2025V4635,Stadt Karlsruhe,2025-08-13T22:00:00Z,"LINESTRING (462945.591 5426090.561, 462971.004...",55,193,Hanggebiet
3,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102305971,geom,...,Nächtliche Vollsperrung vs. ab 0:00 - 5:00 Uhr,mit Vollsperrung,49247,2025V4635,Stadt Karlsruhe,2025-08-13T22:00:00Z,"LINESTRING (460746.25 5425573.74, 460772.555 5...",27,194,Bergwald
3,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102305971,geom,...,Nächtliche Vollsperrung vs. ab 0:00 - 5:00 Uhr,mit Vollsperrung,49247,2025V4635,Stadt Karlsruhe,2025-08-13T22:00:00Z,"LINESTRING (460746.25 5425573.74, 460772.555 5...",55,193,Hanggebiet
4,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102310926,geom,...,NaN,mit Verkehrsbehinderung,47814,2025V5276,Stadt Karlsruhe,2025-09-14T22:00:00Z,"LINESTRING (454132.223 5432567.411, 454133.47 ...",47,261,Südlicher Teil
5,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102310942,geom,...,Aufteilung nicht vorgenommen,mit Verkehrsbehinderung,49914,2025V5280,Stadt Karlsruhe,2025-09-17T22:00:00Z,"LINESTRING (457787.744 5429174.079, 457786.709...",58,073,Westlicher Teil
5,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102310942,geom,...,Aufteilung nicht vorgenommen,mit Verkehrsbehinderung,49914,2025V5280,Stadt Karlsruhe,2025-09-17T22:00:00Z,"LINESTRING (457787.744 5429174.079, 457786.709...",65,071,Nördlicher Teil
6,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102312471,geom,...,NaN,mit Vollsperrung,41073,2025V5450,Stadt Karlsruhe,2025-09-21T22:00:00Z,"LINESTRING (460154.588 5427908.678, 460151.356...",54,192,Dornwald-Untermühl


In [23]:
# 4. Add Stadtteil info to gdf_MultiLineString
gdf_MultiLineString_stadtteil = gpd.sjoin(gdf_MultiLineString, gdf_stadtteile_map, how="left", predicate="intersects")

In [24]:
gdf_MultiLineString_stadtteil

,type,totalFeatures,numberMatched,numberReturned,timeStamp,crs_type,crs_properties_name,feature_type,feature_id,geometry_name,...,zusatzinfo,sperrung,projektnummer,vorgangsnummer,datenquelle,stand,geometry,index_right,Stadtteil_Nummer,Stadtteil
0,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102193847,geom,...,Abwicklung in mehreren Bauabschnitten,mit Verkehrsbehinderung,10273,2023V5734,Stadt Karlsruhe,2025-09-10T22:00:00Z,"MULTILINESTRING ((455923.63 5427422.033, 45592...",33,041,Östlicher Teil
1,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102193848,geom,...,Abwicklung in mehreren Bauabschnitten,mit Verkehrsbehinderung,10273,2023V5734,Stadt Karlsruhe,2025-09-10T22:00:00Z,"MULTILINESTRING ((456048.834 5427160.722, 4560...",33,041,Östlicher Teil
2,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102296302,geom,...,"Abwicklung in mehreren Bauabschnitten, Aufteil...",mit Verkehrsbehinderung,45182,2025V3496,Stadt Karlsruhe,2025-09-24T22:00:00Z,"MULTILINESTRING ((454148.833 5429127.463, 4541...",8,051,Mittlerer Teil
3,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102311078,geom,...,Aufteilung nicht vorgenommen,mit Verkehrsbehinderung,46170,2025V5298,Stadt Karlsruhe,2025-09-15T22:00:00Z,"MULTILINESTRING ((464167.538 5422461.625, 4641...",24,211,Stupferich
4,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102312472,geom,...,NaN,mit Vollsperrung,41073,2025V5450,Stadt Karlsruhe,2025-09-21T22:00:00Z,"MULTILINESTRING ((460050.149 5427790.847, 4600...",54,192,Dornwald-Untermühl
5,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102312473,geom,...,NaN,mit Vollsperrung,41073,2025V5450,Stadt Karlsruhe,2025-09-21T22:00:00Z,"MULTILINESTRING ((460247.707 5427734.328, 4602...",54,192,Dornwald-Untermühl
6,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102312474,geom,...,NaN,mit Vollsperrung,41073,2025V5450,Stadt Karlsruhe,2025-09-21T22:00:00Z,"MULTILINESTRING ((460247.305 5427734.642, 4602...",54,192,Dornwald-Untermühl
7,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102312811,geom,...,Abwicklung in mehreren Bauabschnitten,mit Verkehrsbehinderung,40627,2025V5494,Stadt Karlsruhe,2025-09-23T22:00:00Z,"MULTILINESTRING ((453180.078 5430784.28, 45318...",63,064,Rennbuckel


In [25]:
# 5. Add Stadtteil info to gdf_MultiPolygon
gdf_MultiPolygon_stadtteil = gpd.sjoin(gdf_MultiPolygon, gdf_stadtteile_map, how="left", predicate="intersects")

In [26]:
gdf_MultiPolygon_stadtteil

,type,totalFeatures,numberMatched,numberReturned,timeStamp,crs_type,crs_properties_name,feature_type,feature_id,geometry_name,...,zusatzinfo,sperrung,projektnummer,vorgangsnummer,datenquelle,stand,geometry,index_right,Stadtteil_Nummer,Stadtteil
0,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102021977,geom,...,"Abwicklung in mehreren Bauabschnitten, Aufteil...",mit Verkehrsbehinderung,31027,2020V5592,Stadt Karlsruhe,2025-08-10T22:00:00Z,"MULTIPOLYGON (((455568.34 5426039.452, 455574....",29,141,Weiherfeld
1,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102078846,geom,...,"Abwicklung in mehreren Bauabschnitten, Aufteil...",mit Vollsperrung,35548,2021V5742,Stadt Karlsruhe,2025-09-08T22:00:00Z,"MULTIPOLYGON (((454968.865 5426532.349, 454963...",0,132,Bulach
2,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102260990,geom,...,"Abwicklung in mehreren Bauabschnitten, Arbeits...",mit Verkehrsbehinderung,43211,2024V7262,Stadt Karlsruhe,2024-12-05T23:00:00Z,"MULTIPOLYGON (((454826.021 5426886.497, 454829...",30,131,Beiertheim
3,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102261447,geom,...,"Abwicklung in mehreren Bauabschnitten, Aufteil...",mit Sperrung in eine Fahrtrichtung,40847,2024V5284,Stadt Karlsruhe,2025-10-05T22:00:00Z,"MULTIPOLYGON (((455206.909 5428856.375, 455202...",8,051,Mittlerer Teil
3,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102261447,geom,...,"Abwicklung in mehreren Bauabschnitten, Aufteil...",mit Sperrung in eine Fahrtrichtung,40847,2024V5284,Stadt Karlsruhe,2025-10-05T22:00:00Z,"MULTIPOLYGON (((455206.909 5428856.375, 455202...",43,022,Westlicher Teil
3,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102261447,geom,...,"Abwicklung in mehreren Bauabschnitten, Aufteil...",mit Sperrung in eine Fahrtrichtung,40847,2024V5284,Stadt Karlsruhe,2025-10-05T22:00:00Z,"MULTIPOLYGON (((455206.909 5428856.375, 455202...",34,052,Südlicher Teil
4,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102286477,geom,...,"Abwicklung in mehreren Bauabschnitten, Aufteil...",mit Verkehrsbehinderung,46078,2025V2418,Stadt Karlsruhe,2025-08-31T22:00:00Z,"MULTIPOLYGON (((454101.097 5428233.81, 454115....",34,052,Südlicher Teil
5,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102295047,geom,...,Ein Fahrstreifen im Haupttunnel und ein Fahrts...,mit Sperrung in eine Fahrtrichtung,44397,2025V3336,Stadt Karlsruhe,2025-09-29T22:00:00Z,"MULTIPOLYGON (((456718.097 5428861.135, 456725...",14,021,Östlicher Teil
5,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102295047,geom,...,Ein Fahrstreifen im Haupttunnel und ein Fahrts...,mit Sperrung in eine Fahrtrichtung,44397,2025V3336,Stadt Karlsruhe,2025-09-29T22:00:00Z,"MULTIPOLYGON (((456718.097 5428861.135, 456725...",44,011,Nordöstlicher Teil
6,FeatureCollection,270,270,270,2025-10-06T14:15:33.080Z,name,urn:ogc:def:crs:EPSG::25832,Feature,baustellen_vorschau.102310612,geom,...,Restarbeiten,mit Verkehrsbehinderung,40847,2025V5175,Stadt Karlsruhe,2025-09-11T22:00:00Z,"MULTIPOLYGON (((455203.085 5428616.06, 455356....",43,022,Westlicher Teil


### 4. Check if any features intersect with multiple Stadtteile

In [ ]:
# 
def multi_stadtteile(gdf, label):
    # left index = original feature id after sjoin
    counts = gdf.groupby(gdf.index)["Stadtteil"].nunique(dropna=True)
    multi = counts[counts > 1].sort_values(ascending=False)
    print(label)
    print("total rows:", len(gdf), "unique features:", gdf.index.nunique())
    print("features with >1 Stadtteil:", len(multi))
    if len(multi):
        print("sample indices:", list(multi.index[:10]))
    return multi

multi_ml = multi_stadtteile(gdf_MultiLineString_stadtteil, "gdf_MultiLineString_stadtteil")
multi_ls = multi_stadtteile(gdf_LineString_stadtteil, "gdf_LineString_stadtteil")
multi_poly = multi_stadtteile(gdf_Polygon_stadtteil, "gdf_Polygon_stadtteil")
multi_mpoly = multi_stadtteile(gdf_MultiPolygon_stadtteil, "gdf_MultiPolygon_stadtteil")


gdf_MultiLineString_stadtteil
total rows: 8 unique features: 8
features with >1 Stadtteil: 0
gdf_LineString_stadtteil
total rows: 16 unique features: 10
features with >1 Stadtteil: 4
sample indices: [9, 1, 3, 5]
gdf_Polygon_stadtteil
total rows: 127 unique features: 101
features with >1 Stadtteil: 20
sample indices: [5, 4, 1, 9, 11, 16, 34, 37, 39, 40]
gdf_MultiPolygon_stadtteil
total rows: 16 unique features: 12
features with >1 Stadtteil: 3
sample indices: [3, 5, 7]


In [28]:
output_path

'/Users/jiatong_liu/Documents/CorrelAid/lc-rheinmain-mobidta-task/src/../data/mobidata/processed/'

In [29]:
# Save the results
# CSV requires a regular DataFrame; store geometry as WKT.
def save_wkt_csv(gdf, filename):
    df = gdf.copy()
    df["geometry"] = df.geometry.to_wkt()
    df.to_csv(os.path.join(output_path, filename), index=False)

save_wkt_csv(gdf_Point_stadtteil, "baustellen_vorschau_Point_stadtteil.csv")
save_wkt_csv(gdf_Polygon_stadtteil, "baustellen_vorschau_Polygon_stadtteil.csv")
save_wkt_csv(gdf_LineString_stadtteil, "baustellen_vorschau_LineString_stadtteil.csv")
save_wkt_csv(gdf_MultiLineString_stadtteil, "baustellen_vorschau_MultiLineString_stadtteil.csv")
save_wkt_csv(gdf_MultiPolygon_stadtteil, "baustellen_vorschau_MultiPolygon_stadtteil.csv")

/var/folders/v0/n055j8sd1d1dv_bpn3tp_1n40000gn/T/ipykernel_57752/3024692931.py:5: UserWarning: Geometry column does not contain geometry.
  df["geometry"] = df.geometry.to_wkt()
/var/folders/v0/n055j8sd1d1dv_bpn3tp_1n40000gn/T/ipykernel_57752/3024692931.py:5: UserWarning: Geometry column does not contain geometry.
  df["geometry"] = df.geometry.to_wkt()
/var/folders/v0/n055j8sd1d1dv_bpn3tp_1n40000gn/T/ipykernel_57752/3024692931.py:5: UserWarning: Geometry column does not contain geometry.
  df["geometry"] = df.geometry.to_wkt()
/var/folders/v0/n055j8sd1d1dv_bpn3tp_1n40000gn/T/ipykernel_57752/3024692931.py:5: UserWarning: Geometry column does not contain geometry.
  df["geometry"] = df.geometry.to_wkt()
/var/folders/v0/n055j8sd1d1dv_bpn3tp_1n40000gn/T/ipykernel_57752/3024692931.py:5: UserWarning: Geometry column does not contain geometry.
  df["geometry"] = df.geometry.to_wkt()
